# 01 — EDA & Hiểu Dữ Liệu

**Tham chiếu:** `docs/FinalTerm/pipeline_tong_the.md` Phase 1 · ESC/EAS 2025 Focused Update (Mach F. et al., ehaf190).

## Mục tiêu phase
1. Verify số liệu thực tế: **205 control / 95 case** (≠ readme 100/200).
2. Phân tầng Lp(a) theo ESC/EAS 2025 Figure 3 (normal / slight / **elevated** / markedly_elevated).
3. Đếm và phân tích **33 discordant cases** theo định nghĩa chính thức guideline (Box 1, Table 3–4).
4. Biology check: Lp(a) cao ↔ Plaque_echogenicity = Low (vulnerable plaque).
5. Phân bố echogenicity 4-class: None=205 / Low=28 / Inter=40 / High=27.
6. Xuất figures vào `results/figures/` và annotated CSV cho 02/03 notebooks dùng lại.

Mọi con số ở phase này sẽ được trích dẫn xuyên suốt báo cáo cuối kỳ.

## 0. Setup — Local & Google Colab

Cell dưới tự detect môi trường. Khi chạy Colab, nhập đường dẫn project trong Google Drive (vd `My Drive/it2039-xulytinhieuhinhanhykhoa/project`).

In [ ]:
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    # Đổi path này nếu thư mục project trong Drive của bạn ở chỗ khác
    PROJECT_ROOT = Path('/content/drive/MyDrive/it2039-xulytinhieuhinhanhykhoa/project')
    assert PROJECT_ROOT.exists(), f'Không thấy project tại {PROJECT_ROOT} — chỉnh lại path'
    # Cài deps nếu Colab thiếu (Colab có sẵn numpy/pandas/matplotlib/seaborn/sklearn)
    !pip install -q pyyaml >/dev/null
else:
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('IN_COLAB     =', IN_COLAB)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.clinical_rules import (
    annotate_dataframe,
    discordance_summary,
    ECHOGENICITY_CLASSES,
    LDL_C_GOAL_MG_DL,
    LPA_ELEVATED_THRESHOLD_MG_DL,
)
from src.utils import CSV_PATH, FIGURES_DIR

sns.set_theme(style='whitegrid', context='notebook')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.dpi'] = 150
plt.rcParams['savefig.bbox'] = 'tight'

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
print('Figures sẽ lưu vào:', FIGURES_DIR)

df_raw = pd.read_csv(CSV_PATH)
df = annotate_dataframe(df_raw)
print('Shape:', df.shape)
df.head()

## 1.1 Tổng quan dataset

In [ ]:
print('=' * 55)
print('TỔNG QUAN — 300 bệnh nhân')
print('=' * 55)
print(f'Số dòng / cột       : {df.shape[0]} / {df.shape[1]}')
print(f'Thiếu dữ liệu (NaN) : {df.isna().sum().sum()} (đã fill Plaque_echogenicity NaN → "None")')
print()
print('Plaque_present:')
print(df['Plaque_present'].value_counts().rename({0: 'Control (0)', 1: 'Case (1)'}).to_string())
print(f'\nImbalance ratio  : Control:Case = {(df.Plaque_present==0).sum() / (df.Plaque_present==1).sum():.2f} : 1')
print()
print('Sex:')
print(df['Sex'].value_counts().to_string())
print()
print('Baseline_Risk_Category:')
print(df['Baseline_Risk_Category'].value_counts().to_string())
print()
print('Plaque_echogenicity (4-class):')
print(df['Plaque_echogenicity'].value_counts().reindex(ECHOGENICITY_CLASSES).to_string())

In [ ]:
# Phân bố target / sex / risk-category — 3 plot trên 1 figure
fig, axes = plt.subplots(1, 3, figsize=(13, 3.5))

ax = axes[0]
sns.countplot(x='Plaque_present', data=df, ax=ax, hue='Plaque_present', palette=['#4c72b0', '#dd8452'], legend=False)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Control', 'Case'])
ax.set_title('Plaque_present (205 / 95)')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom')

ax = axes[1]
sns.countplot(x='Sex', data=df, ax=ax, hue='Sex', palette='Set2', legend=False)
ax.set_title('Sex distribution')
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()), ha='center', va='bottom')

ax = axes[2]
sns.countplot(x='Baseline_Risk_Category', data=df, ax=ax, hue='Baseline_Risk_Category',
              order=['Low', 'Moderate', 'High', 'Very High'], palette='Blues_r', legend=False)
ax.set_title('Baseline Risk Category (SCORE2)')
for p in ax.patches:
    h = p.get_height();
    if h: ax.annotate(int(h), (p.get_x()+p.get_width()/2, h), ha='center', va='bottom')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_target_dist.png')
plt.show()

In [ ]:
# Histogram Age + 6 lipid biomarkers + IMT_mm
num_cols = ['Age', 'Lp(a)_mg_dL', 'ApoB_mg_dL', 'LDL_C_mg_dL',
            'Triglyceride_mg_dL', 'Total_Cholesterol_mg_dL', 'Non_HDL_mg_dL', 'IMT_mm']
fig, axes = plt.subplots(2, 4, figsize=(15, 7))
for ax, col in zip(axes.flat, num_cols):
    sns.histplot(data=df, x=col, hue='Plaque_present', bins=25, kde=True, ax=ax,
                 palette={0: '#4c72b0', 1: '#dd8452'}, alpha=0.55)
    ax.set_title(col)
    ax.set_xlabel('')
    leg = ax.get_legend()
    if leg: leg.set_title('Plaque'); [t.set_text({'0': 'Ctrl', '1': 'Case'}[t.get_text()]) for t in leg.texts]
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_numeric_dist.png')
plt.show()

In [ ]:
# Mô tả thống kê numeric — copy thẳng vào báo cáo
df[num_cols].describe().round(2)

## 1.2 Phân tầng Lp(a) theo ESC/EAS 2025 (Figure 3 + Box 1)

Ngưỡng chính thức:

| Tier | mg/dL | nmol/L | Ý nghĩa |
|---|---|---|---|
| `normal` | <30 | <62 | Không tăng nguy cơ |
| `slight` | 30–<50 | 62–<105 | Cân nhắc |
| `elevated` | **>50** | **>105** | **Risk modifier — Box 1** |
| `markedly_elevated` | ≥180 | ≥430 | Tự động High risk |

Cột `lpa_tier` đã được tính sẵn ở `annotate_dataframe()`. Kỳ vọng có gradient sinh học tăng dần khi Lp(a) tăng — đúng cơ chế guideline.

In [ ]:
# Cross-tab Lp(a) tier × Plaque
tier_order = ['normal', 'slight', 'elevated', 'markedly_elevated']
ct = pd.crosstab(df['lpa_tier'], df['Plaque_present']).reindex(tier_order, fill_value=0)
ct.columns = ['Control', 'Case']
ct['Total'] = ct.sum(axis=1)
ct['Plaque rate (%)'] = (ct['Case'] / ct['Total'].replace(0, np.nan) * 100).round(1)
ct

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Plot 1: tỷ lệ plaque theo tier — gradient sinh học
ax = axes[0]
rates = ct['Plaque rate (%)'].dropna()
sns.barplot(x=rates.index, y=rates.values, ax=ax, hue=rates.index,
            palette=['#4c72b0', '#a8c5e8', '#dd8452', '#b14d2a'], legend=False)
for i, v in enumerate(rates.values):
    ax.text(i, v + 0.5, f'{v:.1f}%', ha='center')
ax.set_ylabel('% bệnh nhân có plaque')
ax.set_xlabel('Lp(a) tier (ESC/EAS 2025)')
ax.set_title('Plaque rate theo Lp(a) tier — gradient sinh học')
ax.set_ylim(0, max(rates.values) * 1.25)

# Plot 2: phân bố Lp(a) value, đánh dấu ngưỡng 50
ax = axes[1]
sns.histplot(data=df, x='Lp(a)_mg_dL', hue='Plaque_present', bins=30, kde=True, ax=ax,
             palette={0: '#4c72b0', 1: '#dd8452'}, alpha=0.55)
ax.axvline(LPA_ELEVATED_THRESHOLD_MG_DL, color='red', linestyle='--', linewidth=1.5,
           label=f'Elevated threshold = {LPA_ELEVATED_THRESHOLD_MG_DL:.0f} mg/dL')
ax.legend()
ax.set_title('Phân bố Lp(a) — ngưỡng Box 1 = 50 mg/dL')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_lpa_tier_plaque.png')
plt.show()

## 1.3 Discordance theo ESC/EAS 2025 ⭐

**Định nghĩa chính thức:** bệnh nhân *đạt LDL goal cho category hiện tại* (nhìn lipid panel "ổn") NHƯNG có ≥1 risk modifier (Lp(a)>50 hoặc plaque) → cần reclassify.

Tham chiếu: Table 3 + Table 4 + Box 1 của ehaf190.

Kỳ vọng trên 300 bệnh nhân: **33 discordant** (7 chỉ Lp(a) · 22 chỉ plaque · 4 cả hai).

In [ ]:
print('LDL-C goals theo Table 4 (mg/dL):', LDL_C_GOAL_MG_DL)
print()
summary = discordance_summary(df)
print(summary)

n_disc = int(df['is_discordant'].sum())
print(f'\nTổng discordant cases: {n_disc}')
assert n_disc == 33, f'Expected 33 discordant cases, got {n_disc}'
print('✓ Đúng kỳ vọng — verify ESC/EAS 2025 definition')

In [ ]:
# Plot phân bố subtype discordance
fig, ax = plt.subplots(figsize=(9, 4))
order = ['above_ldl_goal', 'discordant_lpa_only', 'discordant_plaque_only', 'discordant_both', 'truly_low_risk']
labels = ['Above LDL goal', 'Discordant: Lp(a) only', 'Discordant: plaque only', 'Discordant: both', 'Truly low risk']
colors = ['#b14d2a', '#dd8452', '#dd8452', '#7a2f1a', '#4c72b0']
counts = df['discordance_subtype'].value_counts().reindex(order, fill_value=0)
bars = ax.barh(range(len(order)), counts.values, color=colors)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(labels)
ax.invert_yaxis()
for i, (bar, n) in enumerate(zip(bars, counts.values)):
    ax.text(n + 2, i, f'n={n} ({100*n/len(df):.1f}%)', va='center')
ax.set_xlabel('Số bệnh nhân')
ax.set_title(f'Phân nhóm theo ESC/EAS 2025  ·  n_discordant = {n_disc}/300 (11.0%)')
ax.set_xlim(0, counts.max() * 1.25)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_discordance_breakdown.png')
plt.show()

In [ ]:
# Liệt kê 33 discordant cases — xem chi tiết
disc_cols = ['Patient_ID', 'Age', 'Sex', 'Lp(a)_mg_dL', 'LDL_C_mg_dL', 'ldl_goal_mg_dl',
             'Baseline_Risk_Category', 'Plaque_present', 'Plaque_echogenicity', 'discordance_subtype']
df_disc = df[df['is_discordant']][disc_cols].copy()
df_disc.sort_values(['discordance_subtype', 'Lp(a)_mg_dL'], ascending=[True, False], inplace=True)
df_disc.reset_index(drop=True, inplace=True)
df_disc

## 1.4 Biology check — Lp(a) cao ↔ Echogenicity Low (vulnerable plaque)

Theo logic sinh học được thiết kế trong dataset (biologically-constrained synthesis):

```
Lp(a) cao  →  Plaque echogenicity = Low (echolucent)
                  ↓
              Mảng bám giàu lipid, mỏng fibrous cap
                  ↓
              Nguy cơ vỡ → đột quỵ cao hơn
```

Nếu correlation này thực sự có trong dữ liệu, thì:
- Nhánh **tabular (MLP)** có thể học gián tiếp echogenicity từ Lp(a)
- Nhánh **imaging (CNN)** nhìn echogenicity trực tiếp từ ảnh
- **Fusion** sẽ tận dụng cả hai → giá trị lâm sàng của model.

In [ ]:
# Boxplot Lp(a) theo Plaque_echogenicity (chỉ trên 95 case)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

ax = axes[0]
df_case = df[df['Plaque_present'] == 1]
sns.boxplot(data=df_case, x='Plaque_echogenicity', y='Lp(a)_mg_dL', ax=ax,
            order=['Low', 'Intermediate', 'High'],
            hue='Plaque_echogenicity', palette={'Low': '#b14d2a', 'Intermediate': '#dd8452', 'High': '#4c72b0'},
            legend=False)
sns.stripplot(data=df_case, x='Plaque_echogenicity', y='Lp(a)_mg_dL', ax=ax,
              order=['Low', 'Intermediate', 'High'], color='black', alpha=0.4, size=3)
ax.axhline(LPA_ELEVATED_THRESHOLD_MG_DL, color='red', linestyle='--', linewidth=1,
           label=f'Lp(a) elevated threshold = 50')
ax.legend()
ax.set_title('Lp(a) theo Plaque_echogenicity (chỉ 95 case)')

# Bar chart: % echogenicity = Low trong mỗi Lp(a) tier (case only)
ax = axes[1]
df_case_tiers = df_case.groupby('lpa_tier', observed=True)['Plaque_echogenicity'].value_counts(normalize=True).unstack(fill_value=0)
df_case_tiers = df_case_tiers.reindex(['normal', 'slight', 'elevated']).fillna(0)
df_case_tiers = df_case_tiers[['Low', 'Intermediate', 'High']]
df_case_tiers.plot(kind='bar', stacked=True, ax=ax,
                    color=['#b14d2a', '#dd8452', '#4c72b0'])
ax.set_title('Echogenicity distribution theo Lp(a) tier (case only)')
ax.set_ylabel('Tỷ lệ')
ax.set_xlabel('Lp(a) tier')
ax.legend(title='Echogenicity', bbox_to_anchor=(1.02, 1), loc='upper left')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_lpa_echogenicity.png')
plt.show()

In [ ]:
# Tỷ lệ echogenicity Low theo Lp(a) tier — số liệu cụ thể
low_pct = (
    df_case.groupby('lpa_tier', observed=True)
           .apply(lambda g: (g['Plaque_echogenicity'] == 'Low').mean() * 100, include_groups=False)
           .reindex(['normal', 'slight', 'elevated'])
           .round(1)
)
print('% case có echogenicity = Low (vulnerable), theo Lp(a) tier:')
print(low_pct.to_string())
print()
print('Nếu gradient tăng dần (normal → elevated) → biology constraint hoạt động đúng.')

## 1.5 Correlation matrix — biomarkers

In [ ]:
corr_cols = ['Age', 'Lp(a)_mg_dL', 'ApoB_mg_dL', 'LDL_C_mg_dL',
             'Triglyceride_mg_dL', 'Total_Cholesterol_mg_dL', 'Non_HDL_mg_dL',
             'IMT_mm', 'Plaque_present']
corr = df[corr_cols].corr(numeric_only=True)

fig, ax = plt.subplots(figsize=(8.5, 7))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', center=0,
            vmin=-1, vmax=1, square=True, ax=ax, cbar_kws={'shrink': 0.8})
ax.set_title('Correlation matrix — biomarkers + IMT + target')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_correlation_matrix.png')
plt.show()

# Top features tương quan mạnh nhất với Plaque_present
target_corr = corr['Plaque_present'].drop('Plaque_present').abs().sort_values(ascending=False)
print('\nTop features tương quan mạnh nhất với Plaque_present:')
print(target_corr.round(3).to_string())

## 1.6 IMT vs Plaque

Theo Vu et al. (2025), IMT là top-1 SHAP feature cho tabular model. Verify giả thuyết này trên dataset hiện tại.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=df, x='Plaque_present', y='IMT_mm', ax=ax,
            hue='Plaque_present', palette={0: '#4c72b0', 1: '#dd8452'}, legend=False)
sns.stripplot(data=df, x='Plaque_present', y='IMT_mm', ax=ax, color='black', alpha=0.3, size=2.5)
ax.set_xticks([0, 1]); ax.set_xticklabels(['Control', 'Case'])
ax.set_title('IMT_mm: Control vs Case')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_imt_plaque.png')
plt.show()

from scipy import stats
imt_ctrl = df.loc[df.Plaque_present == 0, 'IMT_mm']
imt_case = df.loc[df.Plaque_present == 1, 'IMT_mm']
t, p = stats.mannwhitneyu(imt_ctrl, imt_case, alternative='less')
print(f'IMT control mean = {imt_ctrl.mean():.3f} mm  ·  case mean = {imt_case.mean():.3f} mm')
print(f'Mann-Whitney U (one-sided ctrl<case): p = {p:.2e}')

## 1.7 Echogenicity 4-class distribution

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 3.8))
echo_counts = df['Plaque_echogenicity'].value_counts().reindex(ECHOGENICITY_CLASSES)
colors = ['#4c72b0', '#b14d2a', '#dd8452', '#7a9cc4']
bars = ax.bar(echo_counts.index, echo_counts.values, color=colors)
for bar, n in zip(bars, echo_counts.values):
    ax.annotate(int(n), (bar.get_x() + bar.get_width()/2, bar.get_height()), ha='center', va='bottom')
ax.set_title('Plaque_echogenicity (4-class) — Head 2 output target')
ax.set_ylabel('n bệnh nhân')
ax.set_ylim(0, echo_counts.max() * 1.12)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_echogenicity_dist.png')
plt.show()

print('Imbalance nặng — None = 205, Low/Inter/High mỗi class ~30')
print('→ Cần weighted CrossEntropy cho Head 2 (xem Phase 5)')

## 1.8 Export annotated DataFrame

Lưu lại để 02_baseline / 03_fusion dùng lại — tránh phải gọi `annotate_dataframe()` mỗi notebook.

In [ ]:
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)
annotated_path = RESULTS_DIR / 'carotid_annotated.csv'
df.to_csv(annotated_path, index=False)
print(f'Saved annotated dataset: {annotated_path}')
print(f'  shape: {df.shape}')
print(f'  added cols: ldl_goal_mg_dl, at_ldl_goal, lpa_tier, lpa_elevated,')
print(f'              has_plaque, has_risk_modifier, needs_reclassify,')
print(f'              is_discordant, discordance_subtype')

## Tổng kết Phase 1

| Findings | Giá trị |
|---|---|
| n bệnh nhân | 300 (205 control, 95 case) |
| Imbalance | 2.16 : 1 — Control là majority |
| Discordant (ESC/EAS 2025) | **33** (7 chỉ Lp(a), 22 chỉ plaque, 4 cả hai) |
| Lp(a) gradient → plaque rate | 28.4% → 34.9% → 37.8% (normal/slight/elevated) ✓ |
| Echogenicity 4-class | None=205, Low=28, Inter=40, High=27 → cần weighted CE |
| IMT là predictor mạnh | Verified (p < 1e-…) |

**Figures được lưu vào `results/figures/`:**
- `fig_target_dist.png` — 3 plot: plaque/sex/risk
- `fig_numeric_dist.png` — histograms 8 biomarkers theo plaque
- `fig_lpa_tier_plaque.png` — gradient sinh học Lp(a) → plaque rate
- `fig_discordance_breakdown.png` — 33 discordant breakdown
- `fig_lpa_echogenicity.png` — biology check Lp(a) ↔ echogenicity
- `fig_correlation_matrix.png` — heatmap biomarkers
- `fig_imt_plaque.png` — IMT case vs control
- `fig_echogenicity_dist.png` — Head 2 imbalance

**Output dataset:** `results/carotid_annotated.csv` (15 + 9 derived cols).

→ Sẵn sàng chuyển sang Phase 2/3 (Dataset + Baselines).